# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library. All dataset entities (record sets, fields, etc.) are referenced by their Croissant `@id` for explicitness and reproducibility.

### Dataset Source
The dataset source is provided in Croissant schema format at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset's Croissant metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load metadata
dataset = mlc.Dataset(croissant_url)
# Display dataset name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

List available record sets and fields in the dataset. All references use their Croissant `@id`s for reproducibility.

In [ ]:
# List all record sets with their @id and label
record_sets = dataset.metadata.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"@id: {rs.id}\t label: {rs.label if hasattr(rs, 'label') else rs.id}")

# Show fields and columns for the first record set
if len(record_sets) > 0:
    main_record_set = record_sets[0]  # We'll focus on the first record set
    print(f"\nFields and columns for Record Set @id: {main_record_set.id}")
    for field in main_record_set.fields:
        col_id = field.column.id if hasattr(field, 'column') and hasattr(field.column, 'id') else None
        print(f"  Field @id: {field.id}  (Column: {col_id}, Label: {field.label if hasattr(field, 'label') else field.id})")

## 3. Data Extraction

Load records from the main record set (by `@id`) into a pandas DataFrame. Fields and columns are referenced using their Croissant `@id` values.

In [ ]:
# Choose the main record set to extract
# Use the @id from the overview above. Here we use the first one for demonstration.
record_set_id = dataset.metadata.record_sets[0].id

# (Optional) You can list all record sets' @id if you want more.
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}
for record_set in record_sets_ids:
    df = pd.DataFrame(dataset.records(record_set=record_set))
    dataframes[record_set] = df
    print(f"Loaded records for Record Set: {record_set}, Rows: {len(df)}")

# Display available columns (fields by @id) in the primary DataFrame
print(f"\nFields (@id) in '{record_set_id}':")
print(dataframes[record_set_id].columns.tolist())

# Show first records
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's analyze a selected numeric field, filter by value, normalize it, and inspect key groupings.

> **Note:** All fields are referenced by their Croissant `@id`, as found in section 2 above.

In [ ]:
# Pick a numeric field by its @id -- update as appropriate from field overview
# E.g., suppose '@id': 'accession_peptide_count',
# Check all numeric fields:
df = dataframes[record_set_id]

numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print("Numeric field candidates:", numeric_candidates)

# Let's select a field (e.g., the first available numeric field, or replace with target if known)
numeric_field = numeric_candidates[0] if numeric_candidates else None

if numeric_field:
    # Filtering: keep records with value above threshold
    threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records for {numeric_field} > {threshold}\n", filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() != 0 else 1)
    )
    print(f"\nNormalized {numeric_field} for filtered records:\n", filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping: Try grouping by another field (e.g., a label or type field if exists)
    group_candidates = [col for col in df.columns if df[col].dtype == "object"]
    if group_candidates:
        group_field = group_candidates[0]
        # Only use group if at least two unique values
        if df[group_field].nunique() > 1:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
else:
    print("No numeric field available for EDA.")

## 5. Visualization

Visualize the distribution of the numeric field and relationship between variables. This section uses the field `@id`s and standard Python plotting libraries.

In [ ]:
# Plot distribution if numeric_field is set
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field} (Croissant @id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If a group_field exists, show boxplot
    if 'group_field' in locals() and df[group_field].nunique() < 20:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load, inspect, and begin analyzing the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library via Croissant `@id` referencing. 

* We programmatically explored all record sets, fields, and columns referencing their unique `@id`s for reproducibility.
* Extracted data into pandas DataFrames and performed basic exploratory data analysis and normalization using only schema-linked identifiers.
* Visualized numeric data distributions and explored potential groupings.

You are now ready to move on to custom analysis, advanced visualization, or statistical modeling leveraging the rich, FAIR-compliant structure of Croissant-based datasets!